# EXP-DENSENET121-TRANSFER-5SEEDS-021 — real holdout (4)

**Evaluation only.** Load the same five Validation-locked DenseNet121
checkpoints from EXP-018 and score `RL_Cashew_holdout (4).zip` once. There is
no training and no internal Final Test in this notebook. Output goes to the
new EXP-021 directory; prior experiments remain untouched.

EXP-018 was trained and selected on Split (9). This notebook uses
`CashewData_Split (10).zip` only as the reference for dataset and overlap
audits; it does not retrain on Split (10) or score its internal Test.

Holdout (4) has 120 images across four true classes: 46 anthracnose,
13 healthy, 36 leaf miner and 25 red rust. `not_cashew_leaf` remains a possible
prediction but has zero true support. Macro metrics use four supported true
classes. The ZIP SHA-256, image counts and exact-byte overlap with Split 10
are checked before inference.

Compared with Holdout (3), six healthy images were removed; the remaining
images have identical bytes. Keep the two holdout versions and their hashes
separate when reporting results.

Run all cells in order on Colab. The project shortcut is discovered in the
current account's My Drive. Set `FORCE_REMOUNT_DRIVE=True` if the runtime
still mounts the previous account. Use the path overrides when needed.


In [1]:
# 01. Imports and fixed evaluation protocol
import base64
import csv
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import sys
import time
import zipfile
import zlib
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from tensorflow import keras
from tensorflow.keras import layers

SOURCE_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-018"
EXPERIMENT_ID = "EXP-DENSENET121-TRANSFER-5SEEDS-021"
SEEDS = [42, 123, 2026, 3407, 7777]
CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust", "not_cashew_leaf"]
HOLDOUT_CLASSES = ["anthracnose", "healthy", "leaf_miner", "red_rust"]
SPLIT_ZIP = "CashewData_Split (10).zip"
HOLDOUT_ZIP = "RL_Cashew_holdout (4).zip"
SOURCE_SPLIT_SHA256 = "1eb8c46921de19ab53ef8d50e05d8785f94dcb45a5511dcf32e2e7e01516c6f4"
SPLIT_SHA256 = "b24f0faa5e918fb43d7d16500c6b8234df1cd771cc4a27eb2354051bce84edac"
HOLDOUT_SHA256 = "3b340ea830418638fa610fb2176dcc171739cc7fe0bd8e8430c8ed2a614717cc"
EXPECTED_HOLDOUT_COUNTS = {"anthracnose": 46, "healthy": 13,
                           "leaf_miner": 36, "red_rust": 25}
IMG_SIZE, BATCH_SIZE = 224, 16
NUM_CLASSES = len(CLASSES)
HEAD_DROPOUT = 0.40
WEIGHT_DECAY = 1e-4
FORCE_REMOUNT_DRIVE = False
DRIVE_PROJECT_DIR_OVERRIDE = None
SOURCE_RESULTS_ROOT_OVERRIDE = None
DRIVE_RESULTS_ROOT_OVERRIDE = None
DATASET_ZIP_OVERRIDE = None
HOLDOUT_ZIP_OVERRIDE = None

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("TensorFlow", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


TensorFlow 2.21.0 GPUs [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# 02. Mount the current account and locate source/output directories
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive", force_remount=FORCE_REMOUNT_DRIVE)
    drive_root = Path("/content/drive/MyDrive")
    if DRIVE_PROJECT_DIR_OVERRIDE is not None:
        project_dir = Path(DRIVE_PROJECT_DIR_OVERRIDE).expanduser()
    else:
        direct = drive_root / "Cashew_Leaf_model_result"
        candidates = [direct] if (direct / "Dataset" / SPLIT_ZIP).is_file() else []
        if not candidates:
            candidates = [p for p in drive_root.iterdir() if p.is_dir() and
                          (p / "Dataset" / SPLIT_ZIP).is_file()]
        if len(candidates) != 1:
            raise FileNotFoundError(
                f"Found {len(candidates)} project shortcuts. Set DRIVE_PROJECT_DIR_OVERRIDE.")
        project_dir = candidates[0]
    default_results = project_dir / "training_results" / "current_dataset"
    source_root = (Path(SOURCE_RESULTS_ROOT_OVERRIDE).expanduser() if
                   SOURCE_RESULTS_ROOT_OVERRIDE else default_results)
    results_root = (Path(DRIVE_RESULTS_ROOT_OVERRIDE).expanduser() if
                    DRIVE_RESULTS_ROOT_OVERRIDE else default_results)
    split_archive = Path(DATASET_ZIP_OVERRIDE) if DATASET_ZIP_OVERRIDE else project_dir / "Dataset" / SPLIT_ZIP
    holdout_archive = Path(HOLDOUT_ZIP_OVERRIDE) if HOLDOUT_ZIP_OVERRIDE else project_dir / "Dataset" / HOLDOUT_ZIP
    cache_root = Path("/content/dataset_cache/exp021")
elif os.name == "nt":
    project_dir = Path(r"D:\Study\KhoaLuanTotNghiep\Resource")
    default_results = project_dir / "training_results" / "current_dataset"
    source_root = Path(SOURCE_RESULTS_ROOT_OVERRIDE) if SOURCE_RESULTS_ROOT_OVERRIDE else default_results
    results_root = Path(DRIVE_RESULTS_ROOT_OVERRIDE) if DRIVE_RESULTS_ROOT_OVERRIDE else default_results
    dataset_dir = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset")
    split_archive = Path(DATASET_ZIP_OVERRIDE) if DATASET_ZIP_OVERRIDE else dataset_dir / SPLIT_ZIP
    holdout_archive = Path(HOLDOUT_ZIP_OVERRIDE) if HOLDOUT_ZIP_OVERRIDE else dataset_dir / HOLDOUT_ZIP
    cache_root = project_dir / "dataset_cache" / "exp021"
else:
    raise RuntimeError("EXP-021 supports Colab or Windows; set explicit paths for another runtime.")

source_dir = source_root / SOURCE_ID
out_dir = results_root / EXPERIMENT_ID
aggregate_dir = out_dir / "aggregate"
figures_dir = out_dir / "figures"
for directory in (out_dir, aggregate_dir, figures_dir):
    directory.mkdir(parents=True, exist_ok=True)
probe = out_dir / ".write_probe"
try:
    probe.write_text("ok", encoding="utf-8")
finally:
    probe.unlink(missing_ok=True)
existing_lock = out_dir / "evaluation_lock.json"
if existing_lock.is_file():
    previous = json.loads(existing_lock.read_text(encoding="utf-8"))
    previous_reference = previous.get("reference_split_zip_sha256", previous.get("split_zip_sha256"))
    if previous_reference != SPLIT_SHA256:
        raise RuntimeError(
            "Existing EXP-021 outputs use another reference split. Move that result "
            "directory aside or choose a new results root before running this version.")
if not source_dir.is_dir():
    raise FileNotFoundError(f"EXP-018 result directory not found: {source_dir}")
for path in (split_archive, holdout_archive):
    if not path.is_file():
        raise FileNotFoundError(path)
print("EXP-018:", source_dir)
print("EXP-021:", out_dir)
print("Split ZIP:", split_archive)
print("Holdout ZIP:", holdout_archive)


Mounted at /content/drive
EXP-018: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-018
EXP-021: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-021
Split ZIP: /content/drive/MyDrive/Cashew_Leaf_model_result/Dataset/CashewData_Split (10).zip
Holdout ZIP: /content/drive/MyDrive/Cashew_Leaf_model_result/Dataset/RL_Cashew_holdout (4).zip


## Verify the EXP-018 lock and revised holdout archive

Any archive, manifest or checkpoint mismatch stops before holdout inference.


In [3]:
# 03. Verify recipe, selection lock, five checkpoints and archive identities
config_path = source_dir / "experiment_config.json"
lock_path = source_dir / "validation_selection_lock.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
lock = json.loads(lock_path.read_text(encoding="utf-8"))
if config.get("experiment_id") != SOURCE_ID or lock.get("experiment_id") != SOURCE_ID:
    raise ValueError("Source experiment identity mismatch")
if config.get("dataset", {}).get("archive_sha256", "").lower() != SOURCE_SPLIT_SHA256:
    raise ValueError("Source recipe uses a different dataset ZIP")
if lock.get("dataset_sha256", "").lower() != SOURCE_SPLIT_SHA256:
    raise ValueError("Validation lock uses a different dataset ZIP")
if lock.get("config_sha256", "").lower() != sha256_file(config_path):
    raise ValueError("Source config changed after Validation lock")
if config.get("dataset", {}).get("class_names") != CLASSES or lock.get("class_names") != CLASSES:
    raise ValueError("Class order changed")
if config.get("seeds") != SEEDS or lock.get("seeds") != SEEDS:
    raise ValueError("Five-seed protocol changed")
if lock.get("dataset_manifest_sha256") != config["dataset"]["manifest_sha256"]:
    raise ValueError("Source manifest fingerprint mismatch")
if sha256_file(split_archive) != SPLIT_SHA256 or sha256_file(holdout_archive) != HOLDOUT_SHA256:
    raise ValueError("Split 10 or Holdout 4 archive SHA-256 mismatch")
checkpoint_paths = {seed: source_dir / f"seed_{seed}" / "best.weights.h5" for seed in SEEDS}
checkpoint_hashes = {}
for seed, path in checkpoint_paths.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    actual = sha256_file(path)
    if actual.lower() != lock["selected_checkpoint_sha256"][str(seed)].lower():
        raise ValueError(f"Checkpoint changed: seed {seed}")
    checkpoint_hashes[str(seed)] = actual
deployment_seed = int(lock["deployment_seed"])
if deployment_seed not in SEEDS:
    raise ValueError("Invalid deployment seed in source lock")
validation_csv = source_dir / "aggregate" / "validation_results_5seeds.csv"
if sha256_file(validation_csv) != lock["validation_sha256"]:
    raise ValueError("Validation results changed after lock")
validation_rows = pd.read_csv(validation_csv)
if validation_rows["seed"].tolist() != SEEDS:
    raise ValueError("Source Validation results are incomplete")
print("Source lock PASS; deployment seed:", deployment_seed)


Source lock PASS; deployment seed: 123


In [4]:
# 04. Safe extraction and exact manifest verification
def extract_frozen_zip(archive_path, target_parent, root_name, archive_sha):
    target = target_parent / root_name
    marker = target_parent / ".archive_sha256"
    if target_parent.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != archive_sha:
            raise RuntimeError(f"Unverified extraction cache: {target_parent}")
    else:
        staging = target_parent.with_name(target_parent.name + "_staging")
        if staging.exists():
            raise RuntimeError(f"Partial extraction requires review: {staging}")
        staging.mkdir(parents=True)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                name = member.filename.replace("\\", "/")
                parts = Path(name).parts
                resolved = (staging / name).resolve()
                if not parts or parts[0] != root_name or staging.resolve() not in resolved.parents:
                    raise ValueError(f"Unsafe or unexpected ZIP member: {member.filename}")
            archive.extractall(staging)
        (staging / ".archive_sha256").write_text(archive_sha, encoding="utf-8")
        staging.rename(target_parent)
    if not target.is_dir():
        raise FileNotFoundError(target)
    return target

split_root = extract_frozen_zip(split_archive, cache_root / "split10", "CashewData_Split", SPLIT_SHA256)
holdout_root = extract_frozen_zip(holdout_archive, cache_root / "holdout04", "RL_Cashew_holdout", HOLDOUT_SHA256)
source_manifest_path = source_dir / "dataset_manifest.csv"
source_manifest = pd.read_csv(source_manifest_path, dtype={"sha256": str, "relative_path": str})
needed_columns = {"relative_path", "label", "sha256", "split", "class"}
if not needed_columns.issubset(source_manifest.columns) or len(source_manifest) != 7045:
    raise ValueError("EXP-018 source manifest is missing or incomplete")
manifest_records = [{"relative_path": str(row.relative_path), "label": int(row.label),
                     "sha256": str(row.sha256)}
                    for row in source_manifest.itertuples(index=False)]
manifest_digest = hashlib.sha256(json.dumps(
    manifest_records, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
if manifest_digest != lock["dataset_manifest_sha256"]:
    raise ValueError("EXP-018 source manifest content changed")
reference_hashes = set()
source_paths = set()
for row in source_manifest.itertuples(index=False):
    relative = Path(row.relative_path)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe source manifest path: {row.relative_path}")
    image = split_root / relative
    if not image.is_file() or sha256_file(image) != row.sha256:
        raise ValueError(f"EXP-018 image changed in Split 10: {row.relative_path}")
    reference_hashes.add(row.sha256)
    source_paths.add(relative.as_posix())
reference_files = {p.relative_to(split_root).as_posix(): p
                   for p in split_root.rglob("*") if p.is_file()}
if len(reference_files) != 7051:
    raise ValueError(f"Split 10 expected 7,051 images, found {len(reference_files)}")
expected_new = {f"train/healthy/real_healthy_{number:03d}.jpg"
                for number in (35, 37, 38, 39, 40, 43)}
new_paths = set(reference_files) - source_paths
if new_paths != expected_new or source_paths - set(reference_files):
    raise ValueError(f"Split 10 changes beyond the six expected Train/healthy images: {new_paths}")
for relative in sorted(new_paths):
    reference_hashes.add(sha256_file(reference_files[relative]))
if len(reference_hashes) != 7051:
    raise ValueError("Split 10 contains duplicate image bytes")
val_manifest = source_manifest[source_manifest["split"] == "val"].head(2).copy()
val_manifest["image_path"] = val_manifest["relative_path"].map(lambda x: str(split_root / x))

holdout_rows = []
for name in HOLDOUT_CLASSES:
    folder = holdout_root / name
    files = sorted(p for p in folder.iterdir() if p.is_file())
    if len(files) != EXPECTED_HOLDOUT_COUNTS[name]:
        raise ValueError(f"Holdout class count changed: {name}")
    for image in files:
        with Image.open(image) as opened:
            opened.verify()
        digest = sha256_file(image)
        if digest in reference_hashes:
            raise ValueError(f"Holdout image overlaps Split 10: {image}")
        holdout_rows.append({"relative_path": image.relative_to(holdout_root).as_posix(),
                             "image_path": str(image), "class": name,
                             "label": CLASSES.index(name), "sha256": digest})
holdout_manifest = pd.DataFrame(holdout_rows)
if len(holdout_manifest) != 120 or holdout_manifest["sha256"].duplicated().any():
    raise ValueError("Holdout 4 contains missing or duplicate images")
holdout_manifest.to_csv(out_dir / "holdout_dataset_manifest.csv", index=False)
save_json(out_dir / "dataset_audit.json", {
    "source_training_split_sha256": SOURCE_SPLIT_SHA256,
    "reference_split_zip_sha256": SPLIT_SHA256, "reference_images": 7051,
    "source_manifest_sha256": manifest_digest,
    "split10_added_train_healthy": sorted(new_paths),
    "holdout_zip_sha256": HOLDOUT_SHA256,
    "holdout_counts": EXPECTED_HOLDOUT_COUNTS, "holdout_exact_overlap": 0,
    "limitation": "Near duplicates and shared leaf/tree/session IDs need separate review.",
})
print("Dataset audit PASS: Holdout 4", len(holdout_manifest))


Dataset audit PASS: Holdout 4 120


## Rebuild the EXP-018 model graph and load frozen weights

The model definition is copied from EXP-018. `weights=None` only avoids
redownloading ImageNet weights; the trained checkpoint is loaded immediately.


In [5]:

# ============================================================
# 04. REPRODUCIBILITY
# ============================================================

def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    # Đồng bộ Python / NumPy / TensorFlow theo Keras.
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

    print(f"Global seed set to: {seed}")


try:
    tf.config.experimental.enable_op_determinism()
    print("Deterministic operations: ENABLED")
except Exception as e:
    print("Deterministic operations: unavailable")
    print("Reason:", e)

# 08. ImageNet backbone, frozen BN, and smoke test
import tempfile
from PIL import ImageOps

AUGMENTATION_CONFIG = {"horizontal_flip": True, "rotation_factor": 0.08,
                       "zoom_factor": 0.10, "contrast_factor": 0.10,
                       "brightness": False}

def densenet_preprocessing(inputs):
    x = layers.Rescaling(1.0 / 255.0, name="densenet_rescale")(inputs)
    return layers.Normalization(mean=[0.485, 0.456, 0.406],
                                variance=[0.229**2, 0.224**2, 0.225**2],
                                axis=-1, name="densenet_preprocess")(x)

def build_transfer_model(seed, weights="imagenet"):
    set_global_seed(seed)
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="raw_rgb")
    augmented = keras.Sequential([
        layers.RandomFlip("horizontal", seed=seed),
        layers.RandomRotation(0.08, seed=seed),
        layers.RandomZoom(0.10, seed=seed),
        layers.RandomContrast(0.10, seed=seed),
    ], name=f"train_only_augmentation_seed_{seed}")(inputs)
    x = densenet_preprocessing(augmented)
    backbone = keras.applications.DenseNet121(include_top=False, weights=weights,
                                               input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg")
    backbone.trainable = False
    features = backbone(x, training=False)  # Keep BN moving statistics fixed in both stages.
    x = layers.Dropout(HEAD_DROPOUT, seed=seed)(features)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32",
                           name="disease_output")(x)
    return keras.Model(inputs, outputs, name=f"DenseNet121_Transfer_seed_{seed}"), backbone

def set_backbone_stage(backbone, stage):
    if stage == "stage1":
        backbone.trainable = False
    elif stage == "stage2":
        backbone.trainable = True
        for layer in backbone.layers:
            layer.trainable = (layer.name.startswith("conv5_") and
                               not isinstance(layer, layers.BatchNormalization))
        if not any(layer.trainable and layer.weights for layer in backbone.layers):
            raise RuntimeError("No weighted conv5_* layer was unfrozen")
        if any(layer.trainable for layer in backbone.layers
               if isinstance(layer, layers.BatchNormalization)):
            raise RuntimeError("A backbone BatchNorm layer is trainable")
    else:
        raise ValueError(stage)
    return int(sum(np.prod(v.shape) for v in backbone.trainable_weights))

def compile_stage(model, lr):
    model.compile(optimizer=keras.optimizers.AdamW(learning_rate=lr,
                                                  weight_decay=WEIGHT_DECAY),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=["accuracy"], steps_per_execution=1, jit_compile=False)

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def load_locked_model(seed):
    keras.backend.clear_session()
    gc.collect()
    model, backbone = build_transfer_model(seed, weights=None)
    selection = json.loads((source_dir / f"seed_{seed}" / "stage_selection.json").read_text(encoding="utf-8"))
    stage = selection["selected_stage"]
    if stage not in {"stage1", "stage2"}:
        raise ValueError(f"Unexpected selected stage for seed {seed}: {stage}")
    if selection.get("checkpoint_sha256", "").lower() != checkpoint_hashes[str(seed)]:
        raise ValueError(f"Stage selection/checkpoint fingerprint mismatch: seed {seed}")
    set_backbone_stage(backbone, stage)
    model.load_weights(checkpoint_paths[seed])
    return model

def make_eval_dataset(frame):
    paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["label"].to_numpy(dtype=np.int32)
    def decode(path, label):
        raw = tf.io.read_file(path)
        image = tf.io.decode_image(raw, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE], method="bilinear")
        return tf.cast(image, tf.float32), label
    return tf.data.Dataset.from_tensor_slices((paths, labels)).map(
        decode, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Verify the selected seed model against the source .keras deployment package.
source_deployment = source_dir / "model_package" / "deployment_model.keras"
if not source_deployment.is_file():
    raise FileNotFoundError(source_deployment)
selected_model = load_locked_model(deployment_seed)
reference_model = keras.models.load_model(source_deployment, compile=False)
sample = next(iter(make_eval_dataset(val_manifest)))[0]
if not np.allclose(selected_model(sample, training=False).numpy(),
                   reference_model(sample, training=False).numpy(), atol=1e-5):
    raise RuntimeError("Rebuilt checkpoint disagrees with EXP-018 deployment model")
del selected_model, reference_model
keras.backend.clear_session(); gc.collect()
print("Checkpoint/model parity PASS")


Deterministic operations: ENABLED
Global seed set to: 123
Checkpoint/model parity PASS


In [6]:
# 06. Freeze the holdout evaluation inputs before inference
evaluation_lock = {
    "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
    "source_config_sha256": sha256_file(config_path),
    "source_validation_lock_sha256": sha256_file(lock_path),
    "source_validation_results_sha256": sha256_file(validation_csv),
    "source_manifest_sha256": manifest_digest,
    "source_training_split_sha256": SOURCE_SPLIT_SHA256,
    "reference_split_zip_sha256": SPLIT_SHA256, "holdout_zip_sha256": HOLDOUT_SHA256,
    "selected_checkpoint_sha256": checkpoint_hashes,
    "seeds": SEEDS, "deployment_seed": deployment_seed,
    "class_order": CLASSES, "selection_source": "EXP-018 Validation only",
    "holdout_images": 120, "internal_test": "NOT_RUN",
}
evaluation_lock_path = out_dir / "evaluation_lock.json"
if evaluation_lock_path.exists():
    if json.loads(evaluation_lock_path.read_text(encoding="utf-8")) != evaluation_lock:
        raise RuntimeError("EXP-021 evaluation lock changed; do not mix datasets or checkpoints")
else:
    save_json(evaluation_lock_path, evaluation_lock)
print("Evaluation lock PASS")


Evaluation lock PASS


## External real holdout (4)

The five EXP-018 checkpoints stay fixed. All five class probabilities are
exported per image; macro scores use the four true classes in this holdout.


In [7]:
# 07. Evaluate every frozen checkpoint once and export standard atomic outputs
PROB_COLUMNS = [f"prob_{name}" for name in CLASSES]

def write_matrix(matrix, output, normalized):
    suffix = "normalized" if normalized else "raw"
    pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(
        output / f"confusion_matrix_{suffix}.csv")
    fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
    ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if normalized else max(1, matrix.max()))
    ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=40, ha="right")
    ax.set_yticks(range(NUM_CLASSES), CLASSES)
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, f"{matrix[r,c]:.2f}" if normalized else str(int(matrix[r,c])),
                    ha="center", va="center", fontsize=9)
    ax.set(xlabel="Predicted", ylabel="True", title=f"{output.parent.name} / {output.name}")
    fig.tight_layout(); fig.savefig(output / f"confusion_matrix_{suffix}.png", dpi=180); plt.close(fig)

def write_calibration(predictions, output):
    confidence = predictions["confidence"].to_numpy(dtype=float)
    correct = predictions["correct"].to_numpy(dtype=bool)
    probs = predictions[PROB_COLUMNS].to_numpy(dtype=float)
    truth = predictions["true_class_idx"].to_numpy(dtype=int)
    rows = []
    edges = np.linspace(0, 1, 11)
    for i in range(10):
        mask = (confidence >= edges[i]) & (confidence < edges[i+1] if i < 9 else confidence <= 1)
        n = int(mask.sum())
        rows.append({"bin": i, "n": n, "lower": edges[i], "upper": edges[i+1],
                     "accuracy": float(correct[mask].mean()) if n else np.nan,
                     "mean_confidence": float(confidence[mask].mean()) if n else np.nan})
    bins = pd.DataFrame(rows)
    bins.to_csv(output / "calibration_bins.csv", index=False)
    ece = float(sum(r["n"] / len(predictions) * abs(r["accuracy"] - r["mean_confidence"])
                    for r in rows if r["n"]))
    brier = float(np.mean(np.sum((probs - np.eye(NUM_CLASSES)[truth]) ** 2, axis=1)))
    save_json(output / "calibration_metrics.json", {"ece_10_bins": ece,
              "multiclass_brier": brier, "n": len(predictions),
              "note": "Diagnostic only; no threshold or calibration fit."})
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    used = bins[bins["n"] > 0]
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].plot(used["mean_confidence"], used["accuracy"], "o-")
    axes[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="Confidence", ylabel="Accuracy",
                title=f"Reliability; ECE={ece:.3f}")
    axes[1].hist(confidence[correct], bins=edges, alpha=.7, label="Correct")
    axes[1].hist(confidence[~correct], bins=edges, alpha=.7, label="Incorrect")
    axes[1].set(xlabel="Confidence", ylabel="Images", title="Confidence distribution")
    axes[1].legend(); fig.tight_layout()
    fig.savefig(output / "reliability_confidence.png", dpi=180); plt.close(fig)
    return ece, brier

def write_roc_pr(predictions, output):
    truth = predictions["true_class_idx"].to_numpy(dtype=int)
    probs = predictions[PROB_COLUMNS].to_numpy(dtype=float)
    rows = []
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for index, name in enumerate(CLASSES):
        binary = (truth == index).astype(int)
        if binary.sum() in (0, len(binary)):
            rows.append({"class": name, "support": int(binary.sum()), "roc_auc": np.nan,
                         "average_precision": np.nan, "status": "UNDEFINED"})
            continue
        fpr, tpr, _ = roc_curve(binary, probs[:, index])
        precision, recall, _ = precision_recall_curve(binary, probs[:, index])
        auc_score = float(roc_auc_score(binary, probs[:, index]))
        ap = float(average_precision_score(binary, probs[:, index]))
        rows.append({"class": name, "support": int(binary.sum()),
                     "roc_auc": auc_score, "average_precision": ap, "status": "OK"})
        axes[0].plot(fpr, tpr, label=f"{name} {auc_score:.3f}")
        axes[1].plot(recall, precision, label=f"{name} {ap:.3f}")
    axes[0].plot([0, 1], [0, 1], "--", color="gray")
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC OVR")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="PR OVR")
    for ax in axes:
        ax.legend(fontsize=8); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(output / "roc_pr_ovr.png", dpi=180); plt.close(fig)
    pd.DataFrame(rows).to_csv(output / "roc_pr_ovr_metrics.csv", index=False)

def score_split(split_name, frame, seed, model, output):
    dataset = make_eval_dataset(frame)
    _ = model.predict(dataset.take(1), verbose=0)
    started = time.perf_counter()
    probabilities = np.asarray(model.predict(dataset, verbose=0))
    elapsed = time.perf_counter() - started
    y_true = frame["label"].to_numpy(dtype=int)
    if probabilities.shape != (len(frame), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError("Prediction matrix has wrong shape or non-finite values")
    if not np.allclose(probabilities.sum(axis=1), 1, atol=1e-4):
        raise RuntimeError("Prediction probabilities do not sum to one")
    y_pred = probabilities.argmax(axis=1)
    supported = [CLASSES.index(x) for x in HOLDOUT_CLASSES]
    report = classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                                   target_names=CLASSES, output_dict=True, zero_division=0)
    report_frame = pd.DataFrame(report).T
    if split_name == "real_holdout":
        report_frame.loc["not_cashew_leaf", ["precision", "recall", "f1-score"]] = np.nan
    report_frame.to_csv(output / "classification_report.csv")
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    cm_norm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES), normalize="true")
    write_matrix(cm, output, False)
    write_matrix(cm_norm, output, True)
    relative = frame["relative_path"].astype(str).to_numpy()
    predictions = pd.DataFrame({
        "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
        "architecture": "DenseNet121", "training_mode": "ImageNet Transfer, two-stage",
        "dataset_version": "RL_Cashew_holdout_04", "dataset_sha256": HOLDOUT_SHA256,
        "split": split_name, "seed": seed, "image_id": relative, "relative_path": relative,
        "true_label": [CLASSES[i] for i in y_true],
        "predicted_label": [CLASSES[i] for i in y_pred],
        "true_class_idx": y_true, "predicted_class_idx": y_pred,
        "correct": y_true == y_pred, "confidence": probabilities.max(axis=1),
    })
    for index, name in enumerate(CLASSES):
        predictions[f"prob_{name}"] = probabilities[:, index]
    predictions.to_csv(output / "predictions.csv", index=False)
    predictions.loc[~predictions["correct"]].to_csv(output / "error_cases.csv", index=False)
    ece, brier = write_calibration(predictions, output)
    write_roc_pr(predictions, output)
    disease = np.isin(y_true, [CLASSES.index(x) for x in ("anthracnose", "leaf_miner", "red_rust")])
    false_healthy = int(np.sum(disease & (y_pred == CLASSES.index("healthy"))))
    metrics = {
        "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
        "split": split_name, "seed": seed, "n": len(frame),
        "macro_class_count": len(supported), "supported_true_classes": [CLASSES[i] for i in supported],
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=supported,
                                                    average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=supported,
                                              average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=supported,
                                      average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "loss": float(-np.log(np.clip(probabilities[np.arange(len(y_true)), y_true], 1e-7, 1)).mean()),
        "false_healthy_count": false_healthy, "disease_images": int(disease.sum()),
        "false_healthy_rate": float(false_healthy / disease.sum()),
        "ece_10_bins": ece, "multiclass_brier": brier,
        "prediction_seconds_including_input_pipeline": elapsed,
        "inference_ms_per_image_batch16": 1000 * elapsed / len(frame),
        "checkpoint_sha256": checkpoint_hashes[str(seed)],
    }
    save_json(output / "metrics.json", metrics)
    return metrics

all_metrics = {}
for split_name, frame in (("real_holdout", holdout_manifest),):
    rows = []
    for seed in SEEDS:
        output = out_dir / f"seed_{seed}" / split_name
        marker = output / "evaluation_completed.json"
        if marker.is_file():
            saved = json.loads(marker.read_text(encoding="utf-8"))
            if saved.get("evaluation_lock_sha256") != sha256_file(evaluation_lock_path):
                raise RuntimeError(f"Completed {split_name} seed {seed} has a different lock")
            metrics = json.loads((output / "metrics.json").read_text(encoding="utf-8"))
            if not (output / "predictions.csv").is_file():
                raise RuntimeError(f"Completed {split_name} seed {seed} is missing predictions")
            print("Reusing completed", split_name, seed)
        else:
            if output.exists() and any(output.iterdir()):
                raise RuntimeError(f"Partial evaluation requires inspection: {output}")
            output.mkdir(parents=True, exist_ok=True)
            model = load_locked_model(seed)
            metrics = score_split(split_name, frame, seed, model, output)
            del model
            keras.backend.clear_session(); gc.collect()
            save_json(marker, {"evaluation_lock_sha256": sha256_file(evaluation_lock_path),
                               "checkpoint_sha256": checkpoint_hashes[str(seed)],
                               "split": split_name, "seed": seed,
                               "completed_at_utc": datetime.now(timezone.utc).isoformat()})
        rows.append(metrics)
    if [row["seed"] for row in rows] != SEEDS:
        raise RuntimeError("Five-seed evaluation incomplete")
    all_metrics[split_name] = pd.DataFrame(rows)
    all_metrics[split_name].to_csv(aggregate_dir / f"{split_name}_results_5seeds.csv", index=False)
print("Holdout 4 inference complete")


Global seed set to: 42


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 123
Global seed set to: 2026


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 3407


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Global seed set to: 7777


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Holdout 4 inference complete


In [8]:
# 08. Aggregate summaries and visual panels
for split_name in ("real_holdout",):
    results = all_metrics[split_name]
    metric_names = ["accuracy", "macro_precision", "macro_recall", "macro_f1",
                    "balanced_accuracy", "loss", "false_healthy_rate",
                    "ece_10_bins", "multiclass_brier", "inference_ms_per_image_batch16"]
    summary = pd.DataFrame([{"metric": name, "mean": float(results[name].mean()),
                             "sample_sd": float(results[name].std(ddof=1)),
                             "n_seeds": len(SEEDS), "images_per_seed": int(results.iloc[0]["n"])}
                            for name in metric_names])
    summary.to_csv(aggregate_dir / f"{split_name}_mean_std.csv", index=False)
    per_class = []
    for seed in SEEDS:
        report = pd.read_csv(out_dir / f"seed_{seed}" / split_name /
                             "classification_report.csv", index_col=0)
        for name in CLASSES:
            per_class.append({"split": split_name, "seed": seed, "class": name,
                              "support": int(report.loc[name, "support"]),
                              "precision": report.loc[name, "precision"],
                              "recall": report.loc[name, "recall"],
                              "f1": report.loc[name, "f1-score"]})
    per_class = pd.DataFrame(per_class)
    per_class.to_csv(aggregate_dir / f"per_class_{split_name}_by_seed.csv", index=False)
    per_class.groupby("class")[["precision", "recall", "f1"]].agg(["mean", "std"]).to_csv(
        aggregate_dir / f"per_class_{split_name}_mean_std.csv")
    fig, ax = plt.subplots(figsize=(9, 4.5))
    primary = summary[summary["metric"].isin(["accuracy", "macro_recall", "macro_f1", "balanced_accuracy"])]
    ax.bar(primary["metric"], primary["mean"], yerr=primary["sample_sd"], capsize=5)
    ax.set(ylim=(0, 1.05), ylabel="Score", title=f"{split_name}: mean ± sample SD across five seeds")
    fig.tight_layout(); fig.savefig(figures_dir / f"{split_name}_metrics_mean_seed_sd.png", dpi=180); plt.close(fig)
    heat = per_class.pivot(index="class", columns="seed", values="f1").reindex(CLASSES)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    image = ax.imshow(heat, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(SEEDS)), SEEDS)
    ax.set_yticks(range(NUM_CLASSES), CLASSES)
    ax.set(xlabel="Seed", title=f"{split_name}: per-class F1")
    for r in range(NUM_CLASSES):
        for c in range(len(SEEDS)):
            value = heat.iloc[r, c]
            ax.text(c, r, f"{value:.2f}" if np.isfinite(value) else "N/A",
                    ha="center", va="center")
    fig.colorbar(image, ax=ax, label="F1")
    fig.tight_layout(); fig.savefig(figures_dir / f"{split_name}_per_class_f1_heatmap.png", dpi=180); plt.close(fig)
    for normalized in (False, True):
        suffix = "normalized" if normalized else "raw"
        fig, axes = plt.subplots(1, len(SEEDS), figsize=(27, 5.5))
        for ax, seed in zip(axes, SEEDS):
            matrix = pd.read_csv(out_dir / f"seed_{seed}" / split_name /
                                 f"confusion_matrix_{suffix}.csv", index_col=0).to_numpy()
            ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1 if normalized else max(1, matrix.max()))
            ax.set_xticks(range(NUM_CLASSES), CLASSES, rotation=50, ha="right")
            ax.set_yticks(range(NUM_CLASSES), CLASSES)
            ax.set_title(f"seed {seed}")
            for r in range(NUM_CLASSES):
                for c in range(NUM_CLASSES):
                    ax.text(c, r, f"{matrix[r,c]:.2f}" if normalized else str(int(matrix[r,c])),
                            ha="center", va="center", fontsize=8)
        fig.tight_layout()
        fig.savefig(figures_dir / f"{split_name}_confusion_{suffix}_5seeds.png", dpi=180)
        plt.close(fig)

print("Holdout 4 aggregate tables and figures complete")


Holdout 4 aggregate tables and figures complete


In [9]:
# 09. Report, manifest and archives; source checkpoints included in FULL ZIP
import base64
import zlib

readme = f"""# {EXPERIMENT_ID}

Holdout-only continuation of {SOURCE_ID}. No training, Final Test or holdout-based selection.
Source Validation lock: `{sha256_file(lock_path)}`. Deployment seed {deployment_seed}.
EXP-018 training Split 9 SHA-256 `{SOURCE_SPLIT_SHA256}`.
Split 10 overlap reference SHA-256 `{SPLIT_SHA256}`.
Holdout 4 SHA-256 `{HOLDOUT_SHA256}`.
Holdout 4: 120 images, four true classes. Internal Final Test is NOT_RUN here.
`not_cashew_leaf` holdout
F1 is undefined because it has zero true support.
Relative to Holdout (3), six healthy images were removed and the
remaining image bytes are unchanged; retain separate dataset hashes.

Per-seed outputs: `seed_*/real_holdout/` contains metrics, standard
predictions with five probabilities, errors, reports, confusion CSV/PNG,
calibration ECE/Brier and ROC/PR one-vs-rest. Aggregate summaries and panels
are in `aggregate/` and `figures/`. SD is sample SD across five seeds, not an
image-sampling confidence interval. Prediction timing is batch {BATCH_SIZE}
including data loading after one-batch warm-up, and should not be compared
across different devices as a deployment latency benchmark.

`FULL.zip` adds the five EXP-018 source checkpoints and deployment model;
`ANALYSIS.zip` contains reports, notebook and figures without model binaries.
The source EXP-018 result directory remains unmodified.
"""
(out_dir / "README.md").write_text(readme, encoding="utf-8")
save_json(out_dir / "execution_status.json", {
    "experiment_id": EXPERIMENT_ID, "source_experiment_id": SOURCE_ID,
    "training": "REUSED_EXP018", "final_test": "NOT_RUN", "real_holdout": "COMPLETE",
    "source_validation_lock_sha256": sha256_file(lock_path),
    "evaluation_lock_sha256": sha256_file(evaluation_lock_path),
})
required = ["README.md", "execution_status.json", "evaluation_lock.json", "dataset_audit.json",
            "holdout_dataset_manifest.csv",
            "aggregate/real_holdout_results_5seeds.csv", "aggregate/real_holdout_mean_std.csv"]
for seed in SEEDS:
    for split_name in ("real_holdout",):
        required += [f"seed_{seed}/{split_name}/{name}" for name in
                     ("metrics.json", "predictions.csv", "classification_report.csv",
                      "confusion_matrix_raw.csv", "confusion_matrix_raw.png",
                      "confusion_matrix_normalized.csv", "confusion_matrix_normalized.png",
                      "calibration_metrics.json", "roc_pr_ovr_metrics.csv",
                      "evaluation_completed.json")]
missing = [name for name in required if not (out_dir / name).is_file() or
           (out_dir / name).stat().st_size == 0]
save_json(out_dir / "audit_required_artifacts.json", {
    "experiment_id": EXPERIMENT_ID, "required_count": len(required),
    "missing": missing, "status": "PASS" if not missing else "FAIL"})
if missing:
    raise RuntimeError(f"Required evaluation artifacts missing: {missing}")

NOTEBOOK_SOURCE_ZLIB_B64 = "eNrNfdty20iW4K9g1BHToJuCRF1smW5OhCzJVdqyJYUkV2+3xIBBApTQJgkOAMqiFZrYT9iI3Zf9jH2et439kdkv2XPJOwCK8qVjqqOrRCCRefLkyXPPkw9rw2Q8Lta6Vw/0V1guZslad20S5Z/j7Mt0rb02Scoojsporfvw2F4rsnk+hBZXa3/wjv7r2frh0cnF0cnRZWers355vn9y8e7ofH334ujo8GJ9c6vj/b//9j+9PInG3m02jrN56fk7rWvslv714sXRXTSeR2WaTb1sOl4EL15477Mo9srbxCuiSeKN0rvE+z0apzG1Wh9nw89J7B0m0yI5SUoYl3oa3ibDz7MsnZaFN8qzCQG32dnzomnsFcMsT7xP5+/Dg6i4Tb6EBjDB13T2CcYeJoF3eZtAu7SgHqeZV+ZROk2nN9QJ/Ibek3wKk3mX4r8vk6KEZwBrWsDrMhlk2efAO52XM+j7JksKr8xwJtxf8oWBAqzEaZ4MyyxfvPFmeZrlXnI/S/J0kiD4eTKBYb35tMzmMK040PiSk/oSFQwcYIImmIyhP/gBaLyYjVOY2esWzscAzJsXCc/sE2PhEBY1FK07mwoR44WHvQP+82QE+ADMeCMAEWmgSEoaL7tL8nE0o96ieZyWxRsPuolxyjAefEnQGeDAAB50wisB7TUuEYvGFH/VS+PdAiCdrU0vnUQ30HM0zLMClhdIEGY/T7zhOCpgUl1v5yWAVd7m0XCaFUmbOupse7dAeOXtou1tv/TGSTTyJoCxnGawtQtAxl4+h8G9TwBzOGTSwHafxBrAkN4MhkwHY17DGXyTDolaBwAjwvc1yTOGppjPZlkO3X1AQD3YN3k6LBDtDLJ4D6Nic6ZangBRnve34zPv4tf99a3dl22esjfM5kgRCHByHw3L9cGiTCT2vS9peSvw29nktQDk0k6AQQbJiHA9FYto4Pggm8winD71oDC+3Wp7RXov0SbR/gU3BSAEho3fCMKY8Mag3kSz2wg2ahoDCadDWFaEFCb2W5LM6JvyS6Z4AMBfAA55YvAyzRGVt4I8iwSAi2CeX26TKYyFSMNNmCfFfFwWxjzO51MvGo894mG4E7M8hvWF1TnIxtGA0TrLs7/D5vCKW+hnCKPDnohTIESAAlBA+1esxjwHTAGFDwnvfyy8DwvvMAcGFHgXQPmf3p2eHxyF50cfTj+eXIaH58e/H/UuYS0/eemIEQOfwS7maZQpgDbhFSwJkOQuzeaF7D/wPgJp0JsIlgHhyQF/Bc97miQx7/3+Y9tmzsMsTmCA5D6B6QAeQ+purTudj8cuv86IGSGD79vMe7MTeMcTxC0vwyi9B2wkmh0D3spsmI3FGmNLbwAc4OWO+WRY3Jk/b4bmL1zUcTowH/29yKbm76wwf83GUQlUOzGf5QBdZj0pbmHaFlzFwupGrYH4DZxtlI7tRxIuEhaArwQ/8sRb+btNXX3Npolui4sFH8umZ/BTU6R4OInK2TgroVkwW+BfyFJn49JsM51PZsRqpzMLBTBdeIjtY2tOIPCyfDTOvhB7HmmAzo7fS2COcSu2+T+ns0K3KT4DX8ungWRKor0PpDgHrrkIiTG3PdjDOXwbAq0OU9yj4gX15C37ZxCNI+Ayceh2STwuHQFTIFrl/bxCh8NsOpoTCIDNPL1ve6OO7FODB38ABwhhyLtVoHTn5Ynv5a9sGEbzofmTem5pTBrrIJD4GVBW1DUI6I1sNo4WwPY0qVycfkRucnzo9bzrtSd1qc7etdIBjs6PPxwBB1r1W9CRxJj4G7652tlqg2Tdbntbm1sgbbZ3Nl+1vVfwT58FxPv9i4sjanm9ZkjW67U2jCfkA/9AeRmSXOXfwFNDlKr8yxGs12vc/6+n7w9PP16GP2Qc7vLi7P3xZYgyFFHSqOEIREgAZPtG1fDaWiweBIQ0yGj6rpMM9oY7L19vdeKk8zoa7G4no714dzPZ3I33Xu3tjl7vxMPBzm60u9vpxMPR9laylbxKNju7nZfDl6Md2b3b72BrZ7Q5iqLd5HVnbzTY2Y5fxZ2Xu5ubw5eDva3tnXjUGcavXnWGw51o61Uy2Nre3dnc7QyGyd5OEkdDZ5pGz9sDWOwk2tve3OnsvdzeG0UvO5ujwVbn1ct4OOy86rzafj0cvholm4N4D3rb3hzuJfEWtNqBd8OhQYUHl0eHoVpKlIm4kg/OSqJ2Zq1m1+tsP7lV7QXvggZnL3oXNLhH6uT4wy/hxfHfjtre2/3Lg1/pbwBja2sHSPwlNTn5+MEgtXEy9cUv3te/Hu0fgjQ/PYN5wPvNYIe1qb8cHf/yK8j5o4P9v8LzTrLOoq9GDYDX76JxwWKCnoRn56f/BVAUHh6fh6e/H52fHx9isxMpTARJnR9dfHx/eRGen55e1rbj3p5utn+5f3FEFF373qD42vf0rzgZgXSNgFZCFJg+yrpWV61VnN6gydOTgj3gtn5LtSCFEkUifxpkM0A2rNvgeq2FcqsowRicdK3VR+NigGYd6mIpWAX+OJoM4qgrWgfwr9jf8/78Z+BWoKJCX9Bbt0JBDF0wn6H09qlHDRhYJPN8KtvcJvf8l99yJg8CMEQdheBve6gOJcZYpKn1jBlabwLUqkGxm3wGA8/nHwVpiG3Q39OiDLPP9NP57EsO0w7L5L70cewgBtWg8GlssASmqFP3gGGDYJnnSRgVwzTtEbm1vD/Brri+niJPBCU/i0FR7l2vzcvROsgLY3JgZk5LWIlLkk7vQDrhJ+UoCEOhjYch7rBfzj4W4g0K4PQmGCPcs9tFgWp9GIMOO0wKn1rCCK2fp6BugSWF35GK7OjmpLLCCqOZwB9tcEfKtE6FNVHmC718JKBvsuxmnMD0wERQCh9q+dQsuR8ms1Koxkd5nuUG/WMrc9OA2s/P2M6mF07zgCwAwNcG4BN0g3KDxwIcA+EPkxAtK2jRq+EqLburMM+yUlKf2+HGh8Uhd6y/AvCW8KJamFlJIospBFTK0Zq7aQXJPeqsYObmBiNIgDrtXnlhoENjLhtKVIfvid0D4YxDNvSEoFHqIIyBPiCwj0Bb4M76OENfdIx9HbKP4noNfimFoBWkBbOzFoHlXfWtnqEPxIMeoMpa7MFnxLJmyK70XALkXLjtW9jhDMfkX/DpE8LOn60EfAVqFGQaspb3TyCjqrDnUQpzfgddnGTlOyC1mKjarwUKNDRq4j04nT9W7OiCjeJmyghMUqzSle78alNPDZhwBIsvaADRbX6FaJJOOdmE8HUtjfcwllhUXTJ/kJvHJ3peJnxtigZENy3fUglOlObMxhRH9MAGaomkXxkmb5m6sBymAhXlMMqHt8zlGKYatYIovFbfoAHcBaulazWq0LfdcevUFRq3Vo95clzjKzXyMBreNvNU/jikRhuAfDKiGFnJGODIimCK/ukeKtVT1Ei1TK/yT9BhD7vX1xflPF5cX/92m0Xv59H0MitPbm7TZHZ9fZ4wmZp75qfvhae3AmL82WTeROVPErkWWN8wmFyx52BdkciP3wUmOD+X7u2RnqR0l4YsShcEJOldkLspylmcnLOjU0iSNRnXEC7ugv2v6PD/C6iv2ZfijYehA+h2nA7TkrTegqRoBKL3Nsml5zSwlFZBrbyiJulueMp7Qg0RkdzKIrkNz3KWsIv85iZPbkDsiA/kpzht9U7MHNRfULiLmpbijWyH4REZ0kG1wBdt2541HKh8ukfLrhLfrmo5wBoOEgckYUXQKwGXpfvSC9PUuF7LPi+zG0YY5BpXephPx+n0sz9JiwI5jwUXgYpPyZyz4dOuZXobkCd4TarR1pda5zEHF97zHvmQg3EWxYVvf4aGIs+tZk6tSlehDnD11MPgJkHUqFchM4Sv6SxkaxdR5jSuNmlZSnjNgKCrmU4fW22r22OuvIdNJ+auoorClKKIk9pXakQCEk2qOww6RKXYKp6jZ3PfmpqjIo0T3MnD2yzDjj0MZEqRRNtMRJpgC1OwlMKhwqhU+1ko2XpHKwXZZS1VTXV0rQKfAmYNHnY6wqZd70F3/qhoGBV1tNlxU1qsve1yYAMQASzZ5jWkuAxW7RSQFrcAvYt0oyGsabPVoTZiz9gN2IkJPJ27MSdiN5TBPNnUnWSdub5KsP8P3u9Jno4WZIzL1aBdjpY40neRxCrAJ0bTnHx/uvAU4ifRNB2hI4moSsbtPeAok6gc3npFmc0KSVWySyuO+ZM8DtuBnCWGCWYAKgfWMRyGc21zNoKZa4CzlyKcg5+l9Dyw/yQULiNDlgkJKwL+ofCzGPwQx2r47k5lQoQKtgo/5R5tTmlAswqfFAzc6EABtcrnsIPEtJhDGtNNY/QFIv9TARCgA5rCCm1dbvE7OsmkGnJBuDKSKeSSLBRxGfzIAlCpym3v4bElHoqFNRg/uR4BIV/IDNOA1fPyZhCZviglAzhqnI6IuEuVZIHam4bUQI7U1n4gSDq5hjf084EStLUEJtOpbFBiaxVcCWoe3kbTG8x3GZUg1xygn7WsFJEM0XorBHnJ4IBNiI3tlkJ9gF+JTAQBcxN0RZLEsmsOzdkAVN4vHfgdkOo6fqJC9zXj15CSZMdaecHxGNArA4V9+FFp219lBRXDB5XyJslJVtXtSJNKbBnn6kuUS2S0dsUctreDX8vhFEk02K2UoDuKr4uknArEWg6EbND0QAcB/Hdtpj3ihQzp3aOwsAYYhviSpDe3ZRHc7sJTVFVo9UBVocV+dAfhHBkc5VEpN/hFW6k4LkTolJwU/o9WbbBVNCzn0djrVeNF5ljcyuQESIBXiBHOWAsNkBVNXRVl7uPMWn35aR2IxhICjg+0JiHIvsv4lHjXgFVwagwIM2KgRUxoNs4WJIuorx7mrfliDs5LAFxRsvsd4lyu61JKPJ6ShDc6UETBRGWxO0MbGBZ3VW3BMGhd7UEayrvEZALM41mr24f2EOYSmqrIStzA4NnSfrCZesPU8uwLOb9i1jwADBcqCbjz0RXvPIArKDOMY/krslLBt2oAjiilbphNZuOkVN46pabrJfLOQFS8cdeRdHKHOH5iCG0n8C6iEWpEZR6xHqtyCTVXvkOdV+TnqEio+CQc5dnXZIrGrS/VIQ6NllEOciRkX0WbLEGSlW3PUJuMfcvtYR2tD4Es1ZeqKRokSV7T9HotcHUyk9tYzQPyDRQu6xAMkIcwIkPAS8WzpeptAJwCUEF0ZIDSFHaxbHhgUh+njGxMtdNrQq43MGAt+C2OVQ2nFWV0g9a2g6QAI/CETd9+Ts5qjBaH4ks3OoP7nt80YK55VmdRXqYgDIwp5cm/zsFIL8g0TL6gdc6dP7rjykGrri+7HaUWiEy+4G/pDAWURZKUYCAeVCFHYTlJJgOgK+ClolkAVqVgDN3aiAq7+MWHAQ6NT4BEZuNomPgYe4d/SOPdcCemwl5RTs578u/i562AHtU2BpRl4zuSNL5c4g2PvxLv/PphlGTH0cjtAX9cbfaRUtUWI71JIFx1J6WTHFukMRTdxiBTnfz9OC2Q1cAA8ylaYJSPjvlVjDygAAeLj3UIkwsjSCkajyUa7LYGcmqYQsv0cxovm52dLjXmSXUPtVwtit/WeLKWqVH8USUzhR8bvm/Sf4XPvoYZO84sw8WPGKG3nU2mTDcZDp+ayjTDIlXo5iEdJbsyqHi/ucPDVnLq8LGtlLdMN7+yLeo9H665YqgsTgeOrlDXPQhhFLS9B0CVoBlKN+IkszGIQsFUxPPHljjDgUnZIIvH88m04GQ3pznlKkaDZMx/mjZxIZEvTMvrtUfTN2p3DlRVzAcwYRf+QDQgkYVReuc9iaZXmzu7SzUc6cIrHAstLTzh2cfuq5qOwmKeDMHGpXyIh3qc+aCCBdaLloGcLmnS2IQetJZkBDprRB/xk9Zj41fI73NM0pWKs0YgZmqUc5hW4WNy1b3IperbE2xKdjMytAyFxcYKenrzMvycLGQQRxxqyPKiB9hvMxV00V8WEEsCNtyqpKUBYbjgaPW70Xzvf9PCi8C36zLQcRBlfSJNWhtXGr/6xfORb8AsKEYFcStUZLJh+QaZcDQA+TUvhTYHYiEAs4fkmmhDIvFpS1JIMhdDODZIsQo8lhzjczs9z2DfGwoAV35QY1sHNU0vet1i+S0p/mno5QKLE0TCwEqn6qTQk3Nw1zyI4tjcdG42AbsaqJHEdFSEs6xI7/2WS0Y4M+KcMw1BmfkaXy3j4643a9rgKgVLfxnkN+NsADT+An1YMguLcas4LXJMBxrBMXc7SzbOyHASKe3mVRs+Esek2hyG4sQpdwCFXvlpiEE0QMJI5HBsiOzoDTwrGYof4cN0TqrT5nb8GPx9dlMTqpOo4JYU5trebXvbr+D/e/D/121vZxP+vy0wAONa+7WCinVrVZV8Up/9U8+zJoE0a7KB9dpuV8QsUyvGfhbZVBwCTe81wi9NXAnEAzkr6Ky4n2IkxIXw4J2vGlr8pobazW3ozOVK9tvXsQ57zbmfVajKnDrwXzpyGM8xSQITW3kH00E6wytiqjkOc71yfyudo895SvA5OkNuMaF6qwWaxGzhVzuGz2hoIczRJea8dsR9P5hEM5G17d2zkLYY4H3LyOrQiia5dUQ2JhExWiiwWs7xECN/Fz4lx4ClrLJ9pBsJDiPW3MjW5M9rMjWZR1iCBZdUc4eG4w5XOHD/aZ4sfcqk9vGpTu2kfKgYQwgwLz4CjUB0q6YwHfLiBHshKMD8xZ/oZXLZBD8PyPmwcAxIpehUZY/rIBBNSaba1N7kKqjHA09OnGMtTMlEbyqWoUkyQTSDycR+ndbJAtWUKyahmJJlhbNi9i5gqma0GBp8l2hvpd6U4ivIOiD9hz0CbUvJZTQ/2naZbd2gNfcuRxPVxI3FkNwPOUF4i4Ic7rsrPXw/UBwo9gFl04W/PEynAyaKhxkmhMPOFCdzIQjKjKw1M2VItqk3/fhohFQgWSFVxzfMbuTndFJdxMnbYLrJOV1LRValUzLz0gtSE1JtG58vSxbqWqZ222otMdIlOdGuAlTR7buuQWB9xEZ/CAIMhDNNJjROXVVkoPmtRLUNum2t17XnE+rYtoFFto3G5HcOZfEA+GbT7HKcTtKSndDw6nrtJIkM8uFUCwAND23j2bCNMk+SjSIpMM/IOz4syID21OlxdjwGaGzhGI+2p/5QVjNAmiBffVdH/pA8anfQd6TPnCeDeTqOrfwZOu7g3eTR7FYcaIlijx0unogNarGJZ9n5gzgZAZmSmzVFaTJDd7JZ9CLwPonPe3i6Q1Z1uMvSuBDqOAKMw+EuJUlyAsgQH/Ehf1lfwsjQgdHwGzQnJpMkTgHN48XPyschQP8Aesu3/yO6wDDI+dHZ+enhx4Pjt8ewG//6ozrXB8eAv6D1AXoSxnR8jgMD0gzemRVg69+leTYFdnv218tfT09+3b/4FcNRrGWpKKTum3kunoAPVL9aNk7BiGp8WY70yzJ0GqhWf/D+73//j3//H0AGg//49//lnS3KW6CrDe9kPjlbwH/1gTGkisz7Dc80B3oQM8UU/6EzzwEe0C9oXAYhrIFPnLQ6ov8AjTiHkFC8WpDy1gUt4hfCM8dFcQ+XWdcJ8up/WfDpQ206RSgaw6JEg3ESZrMwTkA1nMDmKiZ+yxkYeIZ6W6ZDVKly4lZggRyd7L99j6vYMs+QqZmhZpZ0n9HffBrdRekYwTJVIfnheRIViC9M3jUmDJS+F+jdPIiGnwew/duSpbw9aTMTnWSfYX8DQ7MLDEx0lYTawgKyogBn9H38BdOq9y+PT0+A1Z+8O/6F/aG3WZ5+RUVgHI7GKbF5doGB5MuYv4ejCLM3SQIEm3tLtafrta8Z0I/5RWeTFDAYA0itdF8t72yQI4ubJqy7kerwaO/kGCsLTdHpnyezPBsmpMr46RSZlLGb7/EoMZ3pD86TYhiNsVUn2IQ9s7W7GwCMqNn1ME1AdJhTM1xR2ZsbBBD9nWT5BPr7SsjyJ0k07V1tBjt7YNjDf1CLwDPKL/srqJ3wz12Up1iaAfvY2nr94sUWfr+1taP+2oW/Vuwsuk+L3nqnOjmNLZzfvXOulmQfqiTTApQfPufnc+6KlFRC3Z7ScQyN5jrGahhqhEdYCWY7x/jT9/WRcP0XFrcRMOfRlzC/GZg7K5rfIDOguBt3dZH86xxzB6Oxf2VhRq458bV3QOG+SfOUngsg9gjOdvOX52Iv+LQDVvvmb7ATfKb/ldofiC2y5Ju+wIp0RIWoKoQCHTLP1EheqpIu7oOGPaOwqltLpqTwDNbcWOQeFIFR1QvGGY7nMRpxM1bxNamI/65GsW6YFMFHFXeW9BoJZZZlYw4SRndWoFyCHxCykD9bJ//JeE+iEs94wAvZ2r9vq1pi8uA2CF6qjfT2BLS6O1TDCsQ3ioJCFOMBM3uQlbcUk0wMmWtwnsM8m4H+5JslDMyF9iU0egrylIDuA5HuGzUS2pgBld7RmvTQFhmVk+ieMlcoaHa9NgI1sNzekrr1EnRLNgFWI+jbIQ9ucAiD+TE9fCDWwETWlsBqIjUoJLyU7MSm0LbCe1VFk28oCSLxtYik33aeHD1ivxn92THP+K1IDHRU0O1oa6WOLmWRMNPNS0tGhCE/4UWsul/oudWhz48ogQHgyMsC/Uicvnu3G1KFhifPKvOiYuwCdjj0AnKF+21LcnqLKZKWBGu16jJw0LHgAklGED0TW3zppFfLTwFDMhOcA60ZmusL0SfW0ZtPWT2qyYepA3EZQHW4g26ehaoV57SvOanqRcIlagO62mM1h6jizOFd4G5MVDqL+cQHawOYe+zfBcQ+W4SLOwsPauBQLGDLUQQwngxKpth/pAcAOszDcPQsEO38DBToSfo1yXvMHtTvItiPo8lffKprRUdgwRrtjfNvEgroUyV4wQIYRoueWfml0VU4zopCQIV/Aoe+mEV5kRwAIDegEICud4AFC0EC5uRob+pIFOXqYQ0kUTwL7EHkScmsCMEsCJUd3QPN6+8pul0IPabv61k1TL6j5MgThUYQghGtMIfL/xOVkOn89BIyCgPoJgm5VKmh7BojMt3gvkmmcTBEKg6FL8vAxc0Qsz0wU9p4KDaNoUmtoF2jH8gI3KpjTdYRH//pvHXat/rokfCmtlY9eilEoQbATAWnl7KelyWERYLag5bEbUOYPq4S0lcxRFUrlbtWafc8TZGoaIcgEA41Y3E2pJq2XnPsZVmi+dNAXxCAauQNwxNXd5CiMd19BcXH4b5EvoJ0fPdQwRUO0K/sC/rS3gWT6HMS4rlf6cH3RxiycHgRqqL03I07BmC6gL4JUilvYXiAKiYaG4ECKubXIsLS141ZYQXBBdBvb1mlFeD/lHfDTJKTkNw1wRh9OQrSjOm75oyDmfPBLblb9u370EWbIn1TgLS3jYe5sYxHGE3hPdk8Jgu3OgzEMS9AwNUJLRX/e7vfPLoIfxUgIDlU1fauqjYOiBYQObdZDFt0kIKhA8zH1X5kVuIoGKIBKTqDn0L1bwmUVdIYRwGudSD86wE6kkIuiRgWYyodRRgsxPegIFDw2I5IEgrbmNSAqZcRcMBxiHUai57sf//j5enlx5OjFvBQoH1f135rgZKSjBJ8WGls+sqM86yaJeAGYid7dIORLC4+JTKBRD1HI6V/BtsIUW5VK9CvKzmMXN9IfCU4qnEagDcf15Ncqzs2rZrWnd1pSjitfCxiZZLt8nx7NTKr5qCCmcojP2RZRr8KZh1CArkDt70avaWIMMMQupmi9CDhXeUbZv5Bq9WSFYNk5uQsAOIYghYGzMKals+9V8zvQPCSRm3RmeWT/XhRmY17nWR9t/VErQwOBJVmXAWs4+gmT7D0Lyo/MjxkUBIzV7FHEev2NNsuvNRuqY7xpqpcSPeyPkm1wUsMexDP0GKUjGH4ScdVXgbeO0DDV66KLE9+GzWJhbvPLW3NTne7vgQ5o3Vw0TlU3LUrg7R1yLXSUB09ronPOgdfu43nXGu+Nc4q0dHq+k7UqesnupDHuep7cY5LfX+k+R8ZL39uaHrZ0cJuVSWzP6UDt10+Itb2as74dd3jW+b3fGKYzv4aiR5tBRS5U0Wdp66nczWNg2bofzX9anr+Olugs0UREFlCP6RQDsfMTy/D848nKuhdtzNkfv0zirPUdFBzSAhamqVZ6j5abiZQcqH92dOFUVTxIYNPEAMQCVZvvDjjI1/pvTzKXtilJwpd2ct0kWhDum4ubRdUp6SHA80S5rlaAsHRvbgwofk6DUwRoAoVkq7MShWg6C/YsRx4++OxKGVBCWlY2CdCTRBrV+ARR5mrytcVzBKRjvYGeAJebUDlsbnODDLqyoUM6lIMAeVPLNnxKvAEphNYjyRfyGCnIV7xcg9x7pGrxZdYYz2PUWZPMPLKvdOkz85P34YHp+8/fjihstRghiF2Qpmk55l5imJ/922rh10rXLTcl7XLeQxQaYW3z3IGFPMRLAyVZ9bvMXd9ZLTnYl8UvTLygK1UMDkY50Up7iMOjMgHLZlqZW0tUT6VzH639Hr4wBA+cuKVkaiY3oDmg5DPxmVQzAdY677w4TGaID1/r+29At1oFA1Box9nOWz2L7dpabklo3uwW4rb7IsCfwgGAbR8C7Yw1aK9m6TT3ib+N7rvdeqwAi/8DhaWwQ4C/NVqWSOgLXVfpsPPBZhkwBXMcAcAqFAlI9Q9TJq+jTBYiEawAy/2tniyNzufk/zF1dZVP9ZwlYYCEOKkQwC7jcv2wPO/ytvDfjfYGj3WkxDlMQKPMlq3luRDEhaGGMajyut39s9RBrwF1/q1iyH/ngw8aH3Gt6YgSbe9hXxK8Q2sNJyWYwrtPDAJBsaB1UegR/m4miWb3gQlOW3H0QKDYKDT4jNk2/BffzWKnuFJ2LYXz9JeZ28TukBKZlMCOmnVbW2M9Q84V8PXN8KoaJWxVKQCxrrCmGx6JWqm0KsahwUZ2Ua9giwXtXPdPuh5TQeDLBu3rMJthfO1yeieGh94O2kM9ujI8UPWeNL4vgaI1DwyaWd7UzwivqEoKVhvYzC1Z3igFnYd7ONOx8mF1luis+nshElUoK7vG6j+lx73fZX2W94/W6/+LN/8qUPFg1N48pq3hdmq53Vsd8hUVH7A0QKMhTgxLScxeZBSDmNKVyBQkjCdfEPHIP6S0OHDOXxkPgTAlh6F0xGCrkfL5AsquELY+gFmiwB0tPF5YoDfaTRd2il+FJoUqfuWz57o/tGIkafTwk1S1snJsoWR78u7FMhZ76uQmjSl+RL10J5iKHFB8itCdR96wtRNg1Rb3gsvGhTUwoiveOsePqlMvd9afqaQaBGJGU+fiUENahjkKZ1QYMgAOYQx+C8C6fNOXEekJQubwV/RLgOKffHC22pzok3HFGNWcnMdzkQcychwXgMshZ1NwibR2LA2VR2wAMZjynuZJoCN6Q9FwS5S6/vBy8bYEjlMo5tpxpltYNK8wfvTyltQG1EnJPVbQw5sG3TEtceKYkH8wVItgDtsURlNVjA68Hsn2DWwNC8oiQdnfEX/EmTxL55RZRp7vtrsB9inf4VMB/ai/O/12vo6ok9qLDd5tLAVAONjHK6WitqeeGWF9K7XsvW6vlheppMeccAWiknjh5KkB8YIhijdV0PU73IlYs+TccqKPizI0cFR7wEoohtsjx4rQHX6wS0WRjA4gGA0MA1EbI8YFqzSeAYqQvBKOHQJSiGWnu7y31bt83g6bO7V0jYacHQsrGepcJgN0QEHW2cw5+Tz6gDj5AY5u1AwbKXD0kcqugcdUpE4NyhkJb3D8KvjzuUdWaeR4EVFs/wJZeRHSPHv1SSqWsDKOx1+mPuc9AIUDG1ljiUwMObQJn690gwUFoGBCMqCQEWPe2jJQJM1VeHR4I9Y5tPxyk0+G8DPa/M1bF3APiiEriAukiyPwJsDtDhLli6hok+eEt1CJ3Bv7TK+5ThpOS+YLX88OTx6d3yCucqPVUGHObXp1M07moEUKPFfIRZYlndiCcDbTBNXXSGn+27NfwGSvGqLe6m/wGu1LtUFXUrKWtd2NXbi9DJTnzdcerZiR9+13MuUMoMOjDvJGlY7mrkrffpbZYktyaWXlVkkWmBscz2o4SqiwWSK1ItcVGOdq93N6kXM9wpgi+2Teoi3daZ05jenomeOuVl9LWXB+emBd/r7+ZPC5Zyma3V8ppdB93d27nZHxc7vqfbQvXtcEgxmIWGUOb0HEgEe3+Rp7AtpuPUNpi9T0SwPs7t8ZZFT1dxrlHXdr9I7m9R1nZmENMU+f3E+mDcJxe/bopahCGhVxJesQtprTCtQTZHDcCxVyClfNAxK+NRHlQr20AAm3tu0clNyzgCnSvRglo3YJ2ldbGP7Ssl6jYooBxL1a4c0hzJz8aJZ0TQWnqFnWFT7RUj+1adTHVKnTJEFLuftoZfdp+PNOR89Na0Q1MxFSDUt6JhZ4lt9tDDUWl/ryPHLn+nLctnpQvflfskzzHUmQGis6ToPw3lqhZN344Z3nfkAFxWGEjkOZBB255nw2Ysq4gXQOd7fjHnEBkwL5Lsx36NgQBLlN+h2FMAYnl155y8oPfa533vOn7yvOfDeN7Q/8lj36m/R9Jks2gIomVKB+S5RjWNy1cxIUfSKCt9qLzJvyBBRJmra4NXH8PuOuJ+5kxi6kOir4gmgd63g0squUsyA06S5CoasGuWspdF5MM6GVzW3TLbxPsmZyZixT8m6r9dGnXXiRbCB+ryHQVWqBb/OTVG3FK5nfDjxepUrTJ+3XmZnITpxv7tHI/zQY83fBNmKWwwnOmbheF6cdgSabmwX7jOK+UjOVakd8USWl2FquLT0YBHGDw7yC9Uaa52VYHfOc+nYMG6eJ2JSUXDk/9xGncGTRxTaeP/1ukhwdMeQ59PFnQncRaV6WijrqrklwJtj4caxcMoIMESurIzaFVJXpt4xOuQa1ZZDUy+dcciSVEUOJLe7Svvah8vU2ne/nMkAwZOfY8t+7cCGCdv15K4wu3Ya0HZxOlIuhq4SuD21s2yXfdfh/wbz170+fpe5atrYbhiy7wogbZ7UbZwaLma+XeZnNdsht/23phBEv2aQBEUtsubkSWduW3lOVwuyOOxoiQfEyD2nQ0jM8PFQhuKeS0SzX3MxcfNdxC3DVh3hHGUBBhE/EH5gCck/e77UJ3pOWRDjwtyW6QWWN3ib6U0/jfutyDqUg1holE4vlDQg9h8pubK50o5ahqFMpSvkhuZ8l1pmoJpX+EE1RGLfSe4IzVYDwJZ9zf24HoIG6atA+9bDKJ68iL0ngEE6c9WtBrClniNhNm84/ykAfzuoo44GU17u/kNB/BbQKhfZaxgb7rh/kp7weJDuZR0DntkN8oIhHh+2+bjW3HCHcM/oF5SiE6+hfoVhgZaMxbmjWXxHbzjrMekR4kymzitDDiWe1jqo3L7Zi6JW0OJ3G97ynmoiU0vjUPUqA59PBsEcg2inA8OoiPEZ31k6SzC5nbpne9vtRWWQhhM+aMVKEKWUd0i16mxubnovlL2+sYTNrZZuaJwAMRSF5UE+N7AnflePYPBz7XTBjW8IDOPaCNf7QqLOdy2vdqVQVKtt5jdXvfnuJRbduiQjM/ew5pSRBs6J9Iuq8KaCoVPwZKXa2ExeNBz5bsX3quMecR/bJ6JWKQlfdaOnfAgvlhcaOdmD1j0rZppuXZphQ2Hy+kLsBxIH3oNG4qN5Loh8MOblPuadCzWHEp0DYo1E2foGHKGTZalK2lq2Wt+FBaPMsTFqHR50zZU5NVdkpi9imyrvpf159cCtmLjIaJIZtHyJ2HQhcKHrEz5r1qoIv048VUX4Mb2G04+7Mp+qtvq6AOAZ99crcmk8RFLFi01e3+AUrp7bhLH1uQfzzXPPQLj8gHkxswHOqGjYzN2nd/JKmst3yZGl/a6izK8IodwDYVSG85JCVHhilnzZ0+yLj398xfPg8BI3cTZCvxPg+LE5dlYRarBRrqCBvsTFrOmNX/ZrLnNp9Pbq27Gs8yRufXPSGrXUvNK46i/PaWr4SBrF9rWzJPZM5lR7HU+Nsexe87ijD8F49lR+Ur71XuDty6mghxx2Ripq84FKjTdCzSI861ijaQhr2tEwrEq88p7rBmQ6RjB7qFEBMW2+dp0B166YR23LCmkm/FqDoK2U+naDUrysQ0v1rdd62ysop30jyoDrsHAJFAvzM6bMeDRaDaZ1SDjnQrb1FkUTO6ETcWERN3VWlLEfx9kIs9hW6pFLDRXSO0A7u6Vck4wDya3o8gAeLkjRL7XZl7l4j0+OZebwm6TUb7koXXX7Uv4XTHi5Kw3g5wMXz9GZVQjIvNNiRf3Z21gF70sCGmImeCmFGeBxkVgp1Ky0JzllI0NiRafScwvsVlMtxESQOirJGP3Wal1aTqCaDo33/dV61P6Zmu7kyxX7YgdKTT9GgOuxngAtTqFe1LRdsgVUm9DaDIMFbdPV9kJwk2fz2WDhqxVvXT0VvCNvM8DjX0l2xskvdAVf7emW1YG3d/LqR11eu4moIKoFUxa85Er+V3PlPruhG+RXg6iqF0n9lpneChovOsNTd7y2Zz6MiGq9RZLnPf3cYOx9vONnRhPcrZzv0CmqweZuy8iIuWDKM492GBjuejiw93/+tydOfl8ceji5ouAjafLS05WTXeD/WOGskTOTGiHWFTllEa+cCQMynTivItZZepeV4uISRbD6hJXkYqLoDj7AJUPjmL361aNBzyUrWe5BH5tCIJ86NLX8MJSWtrCM/Mf3HHdyz/9cCJw0kANgd5250ruOmzH1LaemjNlUBRKtC5YfAqSx4gBm3bC/wuEq+tI8VmVkyHCJJXlC72Rjf5kCuOxAlU31lACHO1mU34CVjO51TrSDru/aJpobjjqoypZATitvE1IF9Ckz1PNFxUZyFjgXF377Ycen8oVNrVHtoK1XbW/X2kI6Ha+tFC+63/Iec8+5gyrliBymn6aHEVRPnr20FbK6fAmDgn/yuUpjpGefr9xtPF/p9PocxuN8SvzGFwtTUxepeqxnhVOXzz6m+Q8+rvnMY5t7LXd3NR1vcI84LOcnmowl/UrXhsNU7JSHhrOXVbeHUui8EssNsgdCgPSP8YS8DrxzUrzb+oowcuTy7YjFG1k2yDyEL4raEst59/H9e7wg0yzCPYiK5OWO+eTrOB1oRCDjoZy6EdYauwY4Hqwov1HLWuBqnWr/i8R+UWJi5D2okP9j4J1kqqxN23uXYo2BS5yNvidkHeGKjSJoNEL1lmSq2OB9eqivYPL4KfAOnXutH5xqGo/ctSxgIOESV8S8Vnexf3qoqTECAzBg6jp3vmpCV8cxv6/5UNOXbmcnV1VaUjUO8w4wuxRC4B2LKh0mZtPCE5U6QBHJRf2oT04a4yeJfXr7roNfzad0+QMgbgDGARZggIliSAfD2jywMHG5z3OZf1dm6noLHysc47Va9j1a3heABDA1yTAKFvH1W4LoJmIVjBup6Cbs+VQV2cgTvPpFX78h07XFpVqaLs9A3aOlF1sL6IUE54sN0zO48cm4TYZV+LYq3CB4gs4LpLJJZD1YwfS2RwlIRVuYyEVbJ1B6Bxe/b5yd/MKc1DyMeHRwtPGWcpEQCeenBxtn55gFvH5XrOdY+rHRA2q4PvmecO+T4lMwIWzySfCoDdgLYPjAki6xgtqiUK7gB4D7dWrNcSh1eo3qwIAKGnhGSjOWScUVKzxyGHoPukKauAtPhstpqTx5GQjfxI7T5e++RPlkfT4TNwfcZvMx32Y/YIczXsfCE2bgdWAxTu6w3JvH8Ua97YEgAeoFdADEg4EVgzY+IU8MQBcDZMVxwdVFzIImNSwV4XIrZb3h3vZP9t//9eL4gntU9KSIAU+rDrLssyU8kJZwm3BEi47rpJKAL3UxOAkROyFh3pgGl4G9z7sFtyr0QFd9iwny//BP63ai86P9ww9HwSR2ri1mVt9cbbX+riMt1PgsUO11Rz+4GJZk0iJx9ejjxdFhCF0CeoTHJm2qUlTN9cb7ik8/nL0/ujyyyyD9gLJZ3x27e5Ql8CiuGnPowVjA9rIVaKqy1G66oqqi4z15I1bNF5oBmYhuiDc1Nre9YvrGvga/9TMDP+zmFjj9E1fdMYypDVOz3KirxNOsDfvV7Jna3NflXvAl2naNxQZGq9Ft3VvWgZ/Xq7YLlnRuNFpljCXFA5ac8Fra55KMHHnDski8gHWm5TMDCYoKjPQQwdvonjzrslwXjkpj3IA+/idEWwdzbEUNgHreSZsvlCCEmE+Bd8t8Kw9VHVk5r/KpxZgESqjqHv/pnubk8mESLRKFwkvybv/4vTrsialO/HppIUqqBiCQbUTE1aRVJ96D+Mu+XQl4+NHb09PfQiES/vb++G349uUOOXaEnJPylTBsF5yTr2paivSTmoQT1QovLsU0brN/rqXK95Dg7RCqHDaxuHBzqxOks8V0YN8ZT+LfynFCg4uK+E6ATRSFzyZZMHi5IwoGN81cukbIOwDmpnH7NY6CyfLwtLAu6IbFwofiHRmo6rLWIR8nsahcNJY1BDEn4QpzHANgV2MsoPPHJ9blj8/K8B0t7+6h6e0jDuTW0Km5i2XkEfDGbRRPwi+8rJTYubx3uqkBmuE6uHjrV+mpoSS+sYJ15fBVsfxOy/sTQnhNnGJpifwG7gM60qIA/qZkeh3bEd4F+6zQXUeqHkt5UlWZWqLf/acpcvqfs8joskqfbg3R59QhFUzcuOzVuB+ZCnfq26wF9VjXntvlOKnmphaZbKPCIxK3uqj+u+P3R+GH/ZPjd0cXl1q7OPj16OC3i48fLgIJ9qNbe0tQqg7uy0NiotqnvohXAPuMO3gBlSC1Q3H/tejSFun2tbkmSbHDSSRN2TUSRSJz3YWztYioxK+tT6pIsqw4g2SINQR/hyVH4fuAqW1/5G/+2H/0PH6AcMNPod6KZDd9g++qLCYaY7ExrDPPTIWv5tNLa6h5ynCRnln4q7wvxV/m5QZSfuo0cj4p3JZuTc74LDiXXFoY4opunLLtqgxN8xxHEPeLrCKgVuhduhKwZ1IhzFI4CTpXOTrPcwhQ+oTsqoathB8G5WRmnY9FJxM8R/oK/pbOsNK6r3pCBH3hWDArDhjlUK2Pz8LDo3fv9y+PDukSE8HGq3HM521xQ5DWbHXfWRbRfcCzVPdUYEkKh1wawhgCaqZvcXECPKO7uXy6ssVag9aqYS/7n0bWYfKOSp0ee67d2iDPkiyr+inWX33xzZPWItUsPdx01coAOau4hCO43UXG0jT/KuQ1tfe/AehGgJuvLXgOkIby8KPBMzQbceWrPKewOnhGqecfC5yhFem64IZHqB7I5fynma3A5vDlxNAHh6HuFjqGUQHAO0VqTGkZjXaY6obUmDcebN1Z+mSETiGHw8djLPm24r1mMOI+f4pVU5QzxhtFMNcYTFE1XSuQq54qM4iZevXmXLr0A6/LVpjCvBQhxXC+Qho4noMNb+vFi63NbtAZPXof0rffVcj70LqZUFYvR3dvYd0O7sTVhJP6zvJ+B95xySegOexAwb5fdW1wzzoQjdEpkWqjAhdtj9IndBCzULcAogTdkEIaI5WFFStEeWLHtPL5NDDDTCyFdlpGFKkaIBKWTGGG7rDMK3YYTashvDeG65097ZGIDvqvW6LsDy1O31oFvA0DozOzZIi/4hSsgmhBPkVYOXHt9TbIzNmCW7YAmnE0vZlHN9hiRi3QaOQv+Pf22iMs73TA5xvWujv6Bx4Qz/K17u7j/wdTaxoG"
notebook_dir = out_dir / "notebook"
notebook_dir.mkdir(exist_ok=True)
notebook_copy = notebook_dir / "densenet121_transfer_5seed_021.ipynb"
source_book = json.loads(zlib.decompress(base64.b64decode(NOTEBOOK_SOURCE_ZLIB_B64)))
for cell in source_book["cells"]:
    if cell["cell_type"] == "code":
        cell["source"] = [line.replace('NOTEBOOK_SOURCE_ZLIB_B64 = ""',
                                       f'NOTEBOOK_SOURCE_ZLIB_B64 = "{NOTEBOOK_SOURCE_ZLIB_B64}"', 1)
                          if line.startswith('NOTEBOOK_SOURCE_ZLIB_B64 = ""') else line
                          for line in cell["source"]]
notebook_copy.write_text(json.dumps(source_book, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")

save_json(out_dir / "analysis_manifest.json", {
    "archive_version": "v1", "experiment_id": EXPERIMENT_ID,
    "source_experiment_id": SOURCE_ID, "source_config_sha256": sha256_file(config_path),
    "source_validation_lock_sha256": sha256_file(lock_path),
    "source_training_split_sha256": SOURCE_SPLIT_SHA256,
    "reference_split_zip_sha256": SPLIT_SHA256, "holdout_zip_sha256": HOLDOUT_SHA256,
    "class_order": CLASSES, "seeds": SEEDS, "checkpoint_sha256": checkpoint_hashes,
})
manifest = []
for path in sorted(out_dir.rglob("*")):
    if path.is_file() and path.name not in {"FILE_MANIFEST.csv", "CHECKSUMS.sha256"}:
        manifest.append({"path": path.relative_to(out_dir).as_posix(),
                         "size_bytes": path.stat().st_size, "sha256": sha256_file(path)})
pd.DataFrame(manifest).to_csv(out_dir / "FILE_MANIFEST.csv", index=False)
(out_dir / "CHECKSUMS.sha256").write_text(
    "\n".join(f"{row['sha256']}  {row['path']}" for row in manifest) + "\n", encoding="utf-8")

allowed_analysis = {".csv", ".json", ".md", ".png", ".txt", ".sha256", ".ipynb"}
for target, include_models in ((results_root / f"{EXPERIMENT_ID}_ANALYSIS.zip", False),
                               (results_root / f"{EXPERIMENT_ID}_FULL.zip", True)):
    temporary = target.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(out_dir.rglob("*")):
            if path.is_file() and (include_models or path.suffix.lower() in allowed_analysis):
                archive.write(path, arcname=(Path(EXPERIMENT_ID) /
                                               path.relative_to(out_dir)).as_posix())
        if include_models:
            for seed in SEEDS:
                archive.write(checkpoint_paths[seed], arcname=(Path(EXPERIMENT_ID) /
                    "source_checkpoints" / f"seed_{seed}" / "best.weights.h5").as_posix())
            archive.write(source_deployment, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "deployment_model.keras").as_posix())
            archive.write(config_path, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "experiment_config.json").as_posix())
            archive.write(lock_path, arcname=(Path(EXPERIMENT_ID) /
                "source_checkpoints" / "validation_selection_lock.json").as_posix())
    with zipfile.ZipFile(temporary) as archive:
        if (archive.testzip() is not None or
                f"{EXPERIMENT_ID}/notebook/{notebook_copy.name}" not in archive.namelist()):
            raise RuntimeError(f"Archive verification failed: {temporary}")
    temporary.replace(target)
    print("Verified archive:", target, f"{target.stat().st_size / 2**20:.1f} MiB")


Verified archive: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-021_ANALYSIS.zip 1.8 MiB
Verified archive: /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-DENSENET121-TRANSFER-5SEEDS-021_FULL.zip 218.8 MiB


## Done

EXP-021 reads the EXP-018 Validation lock and five checkpoints. It writes
only Holdout (4) predictions, four-class summaries, plots, manifests and
FULL/ANALYSIS ZIPs. No training or Final Test is run. Holdout (3) and (4)
remain separate dataset versions.
Split (10) is an overlap reference; EXP-018 remains a Split (9) model.
